# Data Cleaning and Preparation
## DAT540 Introduction to Data Science
## University of Stavanger
### L10

In [ ]:
import numpy as np
import pandas as pd

- **Data Transformation**
- *Removing Duplicates*
- Duplicate rows may be found in a DataFrame for various reasons
- *duplicated* instance method returns a boolean Series indicating whether each row is duplicate or has been observed before
- *drop_duplicates* return a DataFrame where the duplicated array is False
- The *subset* argument can be provided to either of the methods for detecting duplicates based on the filtered columns / rows
- *keep* argument in drop_duplicates is used to retain the "first" (default) or "last" observed value

In [ ]:

adult=pd.read_csv("./data/adult.data.csv",header=None)
adult.shape

In [ ]:
adult.head(2)

In [ ]:
# The names of column indices 
#namesurl = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names'
names = pd.read_table("./data/adult.names.txt", header = None)
print("shape", names.shape)
names.head(5)

In [ ]:
# retrieve only the relevant rows # the last 15 rows
names = names.tail(adult.shape[1])
names


In [ ]:
names = names.reset_index(drop=True) # reset the index and drop the old index
#extract the part of each string value before the colon and returns it as a list of strings.
names = list(names.map(lambda x: x.split(':')[0]).values.flatten()) # split by colon symbol and take the first part
print(names)

In [ ]:
adult.head(2)

In [ ]:
# shift indices in regards to the order of columns in adult
names[-1:], names[:-1] = names[:1], names[1:] # swap the last and first elements
print(names)

In [ ]:
adult.columns = names
adult.head(2)

In [ ]:
print('shape of adults before removing duplicates', adult.shape)
# check for duplicated on adults dataset using instance method duplicated
print('no. of duplicates', adult.duplicated().sum(axis=0))
# Remove the duplicate using drop_duplicates
adult.drop_duplicates(inplace = True)
print('shape of adults after removing duplicates', adult.shape)

- *Transforming Data Using a Function or Mapping*
- Allows transformation based on values in an array, Series, or column in a DataFrame
- The *map* instance method on a Series accepts a function or dict-like object containing a mapping. It returns a maps of the given Series to the values in the dict object
- The map method also accepts a function, that performs an element wise operation for each value in the series returning the mapping

In [ ]:
# Transform gender to lower case
adult['sex'].map(lambda x: x.lower()).head() # convert to lower case

- *Replacing Values*
- *replace* instance method, provides even a more simpler way for performing element wise transformation
- It allows replacement of an encountered value to the specified value in a Series or DataFrame
- The first argument (*to_replace*) looks for the values that has to be replaced. The second argument (*value*), specifies the values that would be replaced with
- The arguments, could be scalar or a list allowing us to replace multiple values with scalar or replacing each value with a different replacement
- A single dict would map the keys as the values to be replaced with the value for each key
- the inplace argument can be provided, to perform operations without making a copy

In [ ]:
# Using replace instance method. Set regex=True
# A regular expression (regex) is a sequence of characters that defines a search pattern, often used for string matching or substitution.
adult.replace(to_replace=['Male', 'Female'], value=['m','f'], regex=True, inplace=False).head(2)

- *Discretization and Binning*
- Continuous data is sometimes preferred to be discretized or separated into "bins" for analysis
- pandas offers a top-level *cut* method to segment and sort data values into bins
  - The first argument is a array-like 1-D object that we want to discretise
  - The second argument *bins* is a int, sequence of scalar or pandas.IntervalIndex. Providing a scaler int, creates the specified number of  equal length bins
  - The *right* bool argument indicates whether bins include the right most edge
  - *labels* array or bool specifies the labels for the returned bins (optional)
  - The *precision* (int, default 3) argument defines the precision at which to store and display the bins labels.
  - cut method returns a *Categorical* object with codes and categories
- *qcut* another top-level method bins the data on sample quantiles based on the distribution of the data

- *Renaming Axis Indexes*
- Axis labels can be transformed similarly using mapping
- Like a Series, the axis indices have a map method
- Alternatively, *rename* instance method can be used on a DataFrame to rename the columns and indices. A list or a dict can be provided to the method along with the inplace arg

In [ ]:
# Let us change the column names of the adult dataset with each name starting with a capital letter. 
# Also strip any whitespaces
# The same can be done for the row indices
adult.columns = adult.columns.map(lambda x: x.title().strip())
adult.head(2)

In [ ]:
# Alternatively the rename instance method can be used with a dict to rename columns or index
adult.rename(columns={'>50K, <=50K.': 'Salary', 'Hours-Per-Week': 'Weekly-Work-Hours'}, inplace=True)
adult.head(2)

In [ ]:
# Create bins for the ages into ageBins
# bins = [2, 3, 5, 12, 18, 21, 34, 50, 80, 100, 1000]
# label = labels=['Infant', 'Toddler', 'Early Childhood ', 'Middle Childhood', 'Late Childhood', 'Teen', 'Early Adulthood', 'Midlife', 'Mature Adulthood', 'Late Adulthood']
# right = True, to create inclusive bins
ageBins = pd.cut(adult['Age'], bins=[2, 3, 5, 12, 18, 21, 34, 50, 80, 100, 1000], right=True, labels=['Infant', 'Toddler', 'Early Childhood ', 'Middle Childhood', 'Late Childhood', 'Teen', 'Early Adulthood', 'Midlife', 'Mature Adulthood', 'Late Adulthood'])
print(ageBins)
# Use value_counts() to display the frequencies
ageBins.value_counts()

In [ ]:
# cut chooses the bins to be evenly spaced according to the values themselves and not the frequency of those values. 
# qcut chooses the bins to be evenly spaced according to the frequencies of the bins
# demonstrate qcut and cut on Age with 4 bins. Use value_counts() to display the frequencies
print('pd.qcut:\n', pd.qcut(adult['Age'], 4).value_counts())
print('pd.cut:\n', pd.cut(adult['Age'], 4).value_counts()) 

- *Detecting and Filtering Outliers*
- Is a matter of applying array operations


In [ ]:
# Perfrom describe on the adult data set to have an overview
adult.describe()

In [ ]:
# From an overview, we notice that someone works only 1 hour and another 99 hours per week. 
# How many of such people are there. Is it the norm?
# Might be that there are more anomalies in the Weekly-Work-Hours Series
# Create bins using pd.cut on Weekly-Work-Hours and perform value_counts() to gain a detailed overview
# use bins=[0, 1, 2, 3, 5, 40, 80, 90, 98, 1000] and right=True
# use map on value_counts to generate the percentage
# is there anomalies?
pd.cut(adult['Weekly-Work-Hours'], bins=[0, 1, 2, 3, 5, 40, 80, 90, 98, 100], right=True).value_counts().map(lambda x: x/adult['Weekly-Work-Hours'].value_counts().sum() * 100)

- *Permutaion and Random Sampling*
- Permuting (randomly ordering) a Series or rows in a DataFrame can be performed using *numpy.random.permutation* top-level method
- Calling *permutation* with the length of the axis required for permuting produces an array of integers indicating new ordering
- *iloc* based indexing or the equivalent *take* instance method can be used to select the required indices or columns
- The *sample* instance method of a DataFrame or Series randomly retrieves the number of specified elements from an axis
  - the *replace* bool arg. specifies requirement for generating samples *with replacement*

In [ ]:
# we will use numpy.random.permutation to generate a seq len 3 of permuted ints between 0 and 3
sampler = np.random.permutation(3)
print('sampler', sampler)
# Use iLoc argument of adult df to sample rows (alternatively we could use take instance method with the sampler)
adult.iloc[sampler]

In [ ]:
# Use sample instance method to retrive 3 random rows from adult
adult.sample(3)

- *Computing Indicators/Dummy Variables*
- Converting a categorical variable into a "dummy" or "indicator" matrix
- Similar to *one hot encoding*
- If a column in a DataFrame has *k* distinct values, a matrix or DataFrame can be derived with k columns containing all 1s and 0s
- The pandas *get_dummies* instance method performs the required operation
  - a *prefix* argument allows to add a prefix to each column name

In [ ]:
# example of get_dummies instance method using adult dataset
# get_dummies converts categorical variable into dummy/indicator variables.
# DROP FIRST=True to avoid the dummy variable trap which occurs when the model is trained on the dummy variables.
adult_dummies = pd.get_dummies(adult, columns=['Sex', 'Race' ], prefix=['gender', 'ethnicity'], drop_first=True)
adult_dummies.head(4)

- **String Manipulation**
- applying string and regular expressions concisely on whole arrays of data
- *String Object Methods*
- The *split* instance method is used to split a string into an array based on a specified delimiter
- *trim* instance method, trims white spaces 
- substrings can be concatinated together using the addition *+* symbol
- the python string method *index* finds the first occurrence of a character in a string
- *index* raises an exception if a string isn't found. Alternatively, the *find* method can also be used that returns *-1* if a index is not found
- string instance method *count* returns the number of times a char appears in a string
- *replace* string instance method substitutes occurrence of one pattern to another
- Built-in python string methods
<img src='images/string_methods.png'>

In [ ]:
val= 'a,b,  guido'
val.split(',')

In [ ]:
pieces=[x.strip() for x in val.split(',')]
pieces

In [ ]:
# the substrings can be concatinated using the join method of a string object
'::'.join(pieces) # pass a list to the join method on the string object


In [ ]:
print(val.index(','))

In [ ]:
print(val.index(':'))# substring is not found

In [ ]:
print(val.find(':'))# substring is not found, it returns -1

In [ ]:
print(val.count(',') )# count occurrences of a particular substring
print(val.replace(',', '::'))


- *Regular Expressions*
- A single expression (*regex*) is a string formed according to the regular expression language
- regex describes a pattern to locate in a text that can be used for many purposes
- python provides the built-in *re* module for applying regex to strings
- The *re* module functions fall into three categories: pattern matching, substitution, and splitting
- *re.compile* compiles a regex to be reusable later. 
- regex methods:
  
  ```python
  regex = re.compile('\s*')
  regex.split(text)
  ```
  
<img src='images/regex_methods.png'>

In [ ]:
import re
text= 'foo    bar\t baz  \tqux'
regex= re.compile(r'\s+') # compile the regex pattern into a regex object
print(regex.split(text)) # use the split method of the regex object

In [ ]:
regex.findall(text) # find all substrings where the regex pattern produces a match and return them as a list

In [ ]:
text= """ Dave dave@google.com
            Steve steve@google.com
            Robert robert@google.com
            James james@google.com
            """
pattern= r'[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}' # regex pattern for matching email addresses
regex= re.compile(pattern, flags=re.IGNORECASE) # compile the regex pattern into a regex object, ignore case.insensitive        
regex.findall(text) # find all substrings where the regex pattern produces a match and return them

In [ ]:
m= regex.search(text) # find the first match in the text
m

In [ ]:
print(regex.sub('REDACTED', text)) # replace all matches with REDACTED

- *Vectorized String Functions in pandas*
- Cleaning up a messy dataset for analysis often requires a lot of string munging and regularization
- string and regex methods applied to missing data using map, will fail on NA (null) values
- Series addresses this by providing array-oriented methods for string operations that skip NA values
- These methods are access through the Series *str* attribute
- Partial listing of vectorized string methods:

<img src='images/vectorizedstr_methods.png' width=400>

In [ ]:
adult['Native-Country'].head(5)

In [ ]:
# using findall method on 'Native-Country' to get the list of all occurrences of regex '\-*' 
#  We use the findall or any other vectorized string methods from the str object of a pandas object 
adult['Native-Country'].str.findall(r'\-*').head(5)

In [ ]:
adult['Native-Country'].str.contains('Cuba').head(5) # check if each string contains the substring 'Cuba'

In [ ]:
adult['Native-Country'].str.extract(r'([A-Z]{1}[a-z]+)').head(5) # extract the first capitalized word in each string

In [ ]:
# Create two DataFrames with MultiIndex 
# Gen a sequence of 9 randn
data1 = pd.DataFrame(np.random.randn(9),
                index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'], [1,2,3,1,3,1,2,2,3]])
# Gen a sequence of 6 randn
data2 = pd.DataFrame(np.random.randn(45).reshape(9,5),
                columns=[['a', 'a', 'b', 'b', 'b'], [1,2,3,1,2]])
# Set index names
data1.index.names = ['lev.0', 'lev.1']
data2.columns.names = ['lev.0', 'lev.1']

# Set col
print('data1:\n', data1)
print('data2:\n', data2)

In [ ]:
adult.to_csv('data/adult_sanitized.csv')